# Ch 27 — Distillation 미니

Teacher (Qwen 2.5-1.5B) 로 합성 라벨 생성 → 필터 → Student (Qwen 2.5-0.5B) SFT. SmolLM2 / Gemma 2 가 실제로 쓴 길의 미니 버전.

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/desty/study-tiny-llm/blob/main/notebooks/part7/ch27_distillation.ipynb)

In [ ]:
# !pip install -q transformers peft datasets accelerate

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")
if device == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

## 1. 컨셉 — "큰 모델이 작은 모델을 가르친다"

| 방식 | 무엇을 학습 | 사용 |
|---|---|---|
| **Soft distillation** | Teacher 의 vocab 확률 분포 (logit) | DistilBERT |
| **Hard distillation** | Teacher 의 텍스트 출력만 | **현대 LLM 표준** |

본 책은 **하드 distillation** — 단순, 코드는 LoRA SFT 와 거의 동일.

**실제 사례**: Gemma 2-2B (큰 Gemma로부터), SmolLM2 (Cosmopedia/Mixtral), Phi-3.5-mini (GPT-4 합성), Llama 3 small (Llama 3-405B)

## 4. Teacher 라벨 생성 (최소 예제)

In [ ]:
# distill_collect.py — Teacher 로 라벨 생성
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch, json, random

teacher_id = "Qwen/Qwen2.5-1.5B-Instruct"   # Teacher
tok_t = AutoTokenizer.from_pretrained(teacher_id)
T = AutoModelForCausalLM.from_pretrained(teacher_id, torch_dtype=torch.bfloat16, device_map="auto")

CHARACTERS = ["토끼 토토", "곰 두두", "할머니", "고양이 미미"]
KEYWORDS = ["당근", "비", "달", "친구", "엄마", "꽃"]

samples = []
N = 50  # 노트북 시연용 50개 (실제로는 5000+)

for i in range(N):
    char = random.choice(CHARACTERS)
    kws = random.sample(KEYWORDS, 2)
    prompt = f"3~5세 어린이용 한국어 동화 한 편. 등장: {char}. 키워드: {kws[0]}, {kws[1]}. 200~400자."
    msgs = [{"role": "user", "content": prompt}]
    ids = tok_t.apply_chat_template(msgs, return_tensors="pt", add_generation_prompt=True)
    ids = ids.to(T.device)
    with torch.no_grad():
        out = T.generate(ids, max_new_tokens=500, temperature=0.8, top_p=0.9, do_sample=True)
    text = tok_t.decode(out[0, ids.shape[1]:], skip_special_tokens=True)
    samples.append({"instruction": prompt, "output": text})
    if i % 10 == 0: print(f"  {i}/{N}")

print(f"생성 완료: {len(samples)} 개")
print("예시 출력:", samples[0]["output"][:100], "...")

## 5. 필터 (실전)

In [ ]:
# distill_filter.py
def passes(text, instruction):
    if len(text) < 150 or len(text) > 600: return False
    if any(w in text for w in ["AI", "GPT", "model", "로봇", "인공지능"]): return False
    if text.count("같은") > 5: return False
    if instruction[:20] in text: return False
    ko = sum(1 for c in text if "가" <= c <= "힣")
    if ko / max(len(text), 1) < 0.5: return False
    return True

filtered = [s for s in samples if passes(s["output"], s["instruction"])]
print(f"전체: {len(samples)}개")
print(f"통과: {len(filtered)}개")
print(f"통과율: {len(filtered)/len(samples):.0%}")

# 필터된 데이터 저장
with open("distill_train_filtered.jsonl", "w") as f:
    for s in filtered: f.write(json.dumps(s, ensure_ascii=False) + "\n")

## 6. Student SFT — Ch 24 그대로

In [ ]:
# Student 학습: Ch 24 의 LoRA SFT 코드 그대로, 데이터만 교체
from transformers import AutoModelForCausalLM, AutoTokenizer, TrainingArguments, Trainer
from peft import LoraConfig, get_peft_model
from datasets import load_dataset
import torch

student_id = "Qwen/Qwen2.5-0.5B-Instruct"  # Student
tok_s = AutoTokenizer.from_pretrained(student_id)
del T  # Teacher 메모리 해제
if torch.cuda.is_available(): torch.cuda.empty_cache()

student = AutoModelForCausalLM.from_pretrained(student_id, torch_dtype=torch.bfloat16, device_map="auto")

lora_cfg = LoraConfig(
    r=16, lora_alpha=32,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    lora_dropout=0.05, bias="none", task_type="CAUSAL_LM",
)
student = get_peft_model(student, lora_cfg)
student.print_trainable_parameters()

# 데이터 로드 (Teacher 가 만든 필터된 페어)
ds = load_dataset("json", data_files="distill_train_filtered.jsonl")["train"]

def fmt(ex):
    msgs = [{"role": "user", "content": ex["instruction"]},
            {"role": "assistant", "content": ex["output"]}]
    text = tok_s.apply_chat_template(msgs, tokenize=False)
    enc = tok_s(text, max_length=512, truncation=True, padding="max_length")
    enc["labels"] = enc["input_ids"].copy()
    return enc

ds = ds.map(fmt).remove_columns(["instruction", "output"])

trainer = Trainer(
    model=student,
    args=TrainingArguments(
        output_dir="distill_out", num_train_epochs=3, learning_rate=1e-4,
        per_device_train_batch_size=4, gradient_accumulation_steps=4,
        warmup_steps=10, lr_scheduler_type="cosine",
        bf16=(device == "cuda"), logging_steps=5, save_steps=100,
    ),
    train_dataset=ds, tokenizer=tok_s
)
trainer.train()
student.save_pretrained("distill_out/adapter")
print("Student 학습 완료")

## 연습문제

1. Qwen 2.5-1.5B → 0.5B distillation 1,000 페어. 통과율 + PPL.
2. 필터 임계 30% / 50% / 80% 비교.
3. Teacher = GPT-4 API 시 품질 향상 정도.
4. Distillation Student vs LoRA SFT (Ch 24) 비교.
5. **(생각해볼 것)** OpenAI 모델 Teacher 시 Student 가 Apache 라이선스가 될 수 있나?

In [ ]:
# 연습 1: 1,000 페어 distillation + 통과율/PPL


In [ ]:
# 연습 2: 필터 임계 30/50/80% 비교
for threshold in [0.3, 0.5, 0.8]:
    # 필터를 더 엄격/느슨하게 조정
    # passes_strict = ... 길이 기준 조정
    pass


In [ ]:
# 연습 4: Distillation vs LoRA SFT 비교
